### Ingest circuits.csv file
1. Read the file using Spark dataframe reader API
2. Add metadata columns
    - Source File
    - Ingestion Timestamp
3. Write toi bronze delta table

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-Common/01.environment-config

In [0]:
%run ../00-Common/02.bronze-helpers

In [0]:
from pyspark.sql import functions as f

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/circuits.csv"
table_name = f"{catalog_name}.{bronze_schema}.circuits"

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
circuits_schema = StructType([
  StructField("circuitId", StringType()),
  StructField("url", StringType()),
  StructField("circuitName", StringType()),
  StructField("lat", DoubleType()),
  StructField("lng", DoubleType()),
  StructField("locality", StringType()),
  StructField("country", StringType()),
])

In [0]:
circuits_df = (
    spark.read
        .format('csv')
        .option('header', True)
#        .option(inferSchema=False)
        .option('mode', 'FAILFAST')
        .schema(circuits_schema)
        .load(source_file))

In [0]:
circuits_final_df = add_ingestion_metadata(circuits_df)

In [0]:
%sql
select * from formula1_incr.bronze.circuits

In [0]:
#display(circuits_final_df)

In [0]:
#circuits_final_df = circuits_final_df.withColumn("batch_id",f.lit(v_batch_id))

In [0]:

#(
#    circuits_final_df
#        .write
#        .format('delta')
#        .mode('overwrite')
#        .partitionBy('batch_id')
#        .option('replaceWhere',f"batch_id='{v_batch_id}'")
#        .saveAsTable(table_name)
#)

In [0]:
write_to_bronze(
    input_df = circuits_final_df,
    target_table = table_name,
    batch_id = v_batch_id
)

In [0]:
#display(spark.table(table_name))

In [0]:
#display(spark.table('formula1_incr.bronze.circuits'))